# Lab Activity 1: Exploratory Data Analysis on Restaurant Reviews

## Loading Data

Restaurant reviews dataset was taken from https://www.kaggle.com/datasets/joebeachcapital/restaurant-reviews.

In [ ]:
import pandas as pd
import re
import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [ ]:
df = pd.read_csv('reviews.csv')
df.head()

In [ ]:
# Print all unique restaurants in the Restaurant Reviews dataset

for restaurant in df['Restaurant'].unique().tolist():
    print(restaurant)

### 1. Select a restaurant whose reviews you will analyze. (1 pt.)

Replace the value of the restaurant variable below.

In [ ]:
restaurant = "La La Land - Bar & Kitchen"

restaurant_df = df[df['Restaurant'] == restaurant]
restaurant_df

## Data Pre-processing

Code data pre-processing steps to prepare the text for analysis.

Note: You are expected to perform sentiment analysis and topic modeling in the EDA portion.

### 2. Data Cleaning. (9 pts.)

Clean your data from unwanted symbols and characters (5 pts.). Comment what types of characters you chose to keep or what characters you chose to remove for each regex code, whichever is more concise (4 pts.).

In [ ]:
cleaned_reviews = []
for review in restaurant_df["Review"]:
    ascii = re.sub(r'[^\x00-\x7f]', r'', review) #filters for ascii characters
    new_lines = re.sub(r'[\n]',' ', ascii) #removes new lines
    punctuation = re.sub(r'[!?.,()*/"1234567890+]',' ',new_lines) #removes certain characters used in reviews
    dashes = re.sub(r'- | -',' ',punctuation) #remove dashes not used to hyphenate words
    u = re.sub(r'\su\s',' you ', dashes) #change u to you
    ambience = re.sub(r'ambience|Ambience','ambiance', u) #ambience and ambiance both appeared as common terms in topic modelling so I just did this so they are the same word
    cleaned_reviews.append(re.sub(r'\s+',' ',ambience)) #remove extra whitespace

cleaned_reviews



### 3. Lowercasing (5 pts.)

Add code for lowercasing the entire dataframe below. If you choose not to lowercase the characters, you can leave the code cell below blank. Explain why lowercasing is needed or not needed for your case.

Answer: Lowercasing is important to reduce redundancy in topic modeling by making sure the same words aren't counted separately. Capitalization is not necessarily useful for sentiment analysis or topic modeling either as there are not a lot of relevant proper nouns that would be used and typically restaurant reviews are more formal so people are less likely to convey emotion using all caps text.

In [ ]:
lower_case_reviews = []
for review in cleaned_reviews:
    lower_case_reviews.append(review.lower())

lower_case_reviews

### 4. Stopwords Removal (5 pts.)

Add code for stopwords removal below. You can choose to use the nltk stopwords or other stopwords database packages. If you choose not to remove stopwords, you can leave the code cell below blank. Explain why stopwords removal is needed or not needed for your case.

Answer: I used stopword removal for both topic modeling and sentiment analysis. It is useful for sentiment analysis because the stopwords tend to make the sentences more neutral. I excluded one stopword from being removed which is 'not' because it is still usefull for changing the meaning and sentiment of a sentence. For topic modelling, it is also important because stopwords are used in almost every sentence and without stopword removal the most relevant words would be articles like 'the'.

Additionally, I also lemmatized the reviews for topic modelling to reduce redundancy of the same words in different forms. I did not sue the lemmatized reviews for sentiment analysis because different forms of the word can be used to convey different sentiments. For example 'tastiest' would be more positive than 'tasty'.

In [ ]:
from nltk.corpus import stopwords


word_tokens = [ review.split() for  review in lower_case_reviews]

stop_words = set(stopwords.words('english'))
stop_words.add('us')
no_stopword_reviews = []
for tokens in word_tokens:
    no_stopword_reviews.append([token for token in tokens if token not in stop_words or token=='not'])

no_stopword_reviews = [" ".join(review) for review in no_stopword_reviews]

no_stopword_reviews


In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

from nltk.corpus import stopwords


word_tokens = [ review.split() for review in no_stopword_reviews]

stop_words = set(stopwords.words('english'))
lemmatized_reviews = []
for tokens in word_tokens:
    lemmatized_reviews.append([lemmatizer.lemmatize(token) for token in tokens if token != 'not'])


lemmatized_reviews = [" ".join(review) for review in lemmatized_reviews]

lemmatized_reviews

## Exploratory Data Analysis 

### 5. What is the average general sentiment value across all restaurant reviews for the restaurant you chose? Compare this to the average review rating. What do these two values tell you? (15 pts.)

Average general sentiment value: 0.502306

Average review rating: 3.19

Insight: The average general sentiment leads more positive at ~0.5. A review rating of 3.19 means that we would expect a rating closer to neutral around 0.1-0.2 but the general sentiment is a lot higher than that. An insight is that most reviewers still list the positives of their experience even if they have a middling opinion of the restaurant, which can cause the sentiment value to lean more positive.

In [ ]:
sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

restaurant_docs = nlp_with_sentiment.pipe(no_stopword_reviews)

restaurant_df_modified = pd.DataFrame(columns=['text', 'document', 'com_sent', 'pos_sent', 'neg_sent', 'neu_sent'])
for doc in restaurant_docs:
    new_row = pd.DataFrame([
        {
            'text': doc.text,
            'document': doc,
            'com_sent': doc._.sentiment['compound'],
            'pos_sent': doc._.sentiment['pos'],
            'neg_sent': doc._.sentiment['neg'],
            'neu_sent': doc._.sentiment['neu']
        }
    ])
    restaurant_df_modified = pd.concat([restaurant_df_modified, new_row], ignore_index=True)
restaurant_df_modified.head()

ave_sent = sum(restaurant_df_modified['com_sent']) / len(restaurant_df_modified['com_sent'])
ave_rating = sum( int(rating) for rating in restaurant_df['Rating'] ) / len(restaurant_df['Rating'])
print("average sentiment: " + str(ave_sent))
print("average rating: " + str(ave_rating))


### 6. Get the average positive, neutral, and negative sentiment value across all restaurant reviews for the restaurant you chose. What do these values tell you? (15 pts.)

Average positive sentiment value: 0.31887

Average neutral sentiment value: 0.5842

Average negative sentiment value: 0.09688

Insight: This means that there are a lot less negative reviews than positive reviews which lines up with the average compound score while most of the reviews lean neutral. This lines up with the average rating given. It can be said that the score was cause by a lot of neutral to positive reviews with a handful of negative reviews dropping the score down. One factor to consider is that reviews use terms such as 'not good' or 'not the best', even if these sentiments are negative they may still be treated as more neutral or positive leaning than they actually are.

In [ ]:
ave_pos_sent = sum(restaurant_df_modified['pos_sent']) / len(restaurant_df_modified['pos_sent'])
ave_neu_sent = sum(restaurant_df_modified['neu_sent']) / len(restaurant_df_modified['neu_sent'])
ave_neg_sent = sum(restaurant_df_modified['neg_sent']) / len(restaurant_df_modified['neg_sent'])

print("average positive: " + str(ave_pos_sent))
print("average neutral:  " + str(ave_neu_sent))
print("average negative: "+ str(ave_neg_sent))

### 7. Perform topic modeling on the reviews and explain the insights you have gathered. (25 pts.)

Do LDA topic modeling on the reviews for the restaurant you have chosen. What insights can you glean from these themes?

15 points is awarded for doing general topic modeling. The other 10 points will be given if you are able to do topic modeling that shows deeper insights into the reviews by manipulating the dataset in a certain way.

Insight: In order to gain deeper insights on the dataset I decided to split the topic modeling between the high positive reviews, reviews with a rating of 4 or higher, and low negative reviews, reviews with a rating of 2 and below.

For the positive reviews, the first topic talks about the logistics and the options with the most salient words being starter, buffet, main, lunch, course being words that describe the meal options they have and words like visit, price, service, time, and staff talking about the good logistics of the restaurant. The second topic talks about the vibe of the restaurant with words like music, drink, friend, loud, dj, hangout, dance, and floor suggesting that this place is a good place to hangout. This topic seems to be reviews that focus on the restaurant as a bar. Finally, the third topic talks about the actual food of the place with salient words including veg, chicken, rice, fish, dinner, paneer. This topic focuses on reviews on the kitchen. This makes sense because the restaurant I chose is both a bar and a restaurant so naturally topics form around its different aspects.

The negative reviews have topics that are much more similar. This makes sense because of the smaller sample size for negative reviews compared to positive reviews. Topic 1 seems to talk about the food with terms like starter, chicken, spicy, menu and taste. Topic 2 covers bad experiences with the staff with terms like manager and staff alongside rude, incorrect, and sorry. The 3rd topic feels more scattered with it containing both words like money bill and price, and words like service, taste, and lunch. This topic seems to be for reviews who do not believe the food is worth the price. Interestingly worlds like 'good' are still part of the most salient terms of the negative reviews which aligns with the lower negative scores seen in the sentiment analysis.

In [ ]:
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel
import pyLDAvis, pyLDAvis.gensim_models

ratings = [int(rating) for rating in restaurant_df['Rating']]

restaurant_df_topic_modeling = pd.DataFrame(columns=['Review', 'Rating'])


for i in range(len(lemmatized_reviews)):
    new_row = pd.DataFrame([
            {
                'Review': lemmatized_reviews[i],
                'Rating': ratings[i],
            }
        ])
    restaurant_df_topic_modeling = pd.concat([restaurant_df_topic_modeling, new_row], ignore_index=True)
restaurant_df_topic_modeling

high_texts = []
for i in range(len(restaurant_df_topic_modeling['Review'])):
    if(restaurant_df_topic_modeling['Rating'][i]>=4 ):
        high_texts.append([word for word in lemmatized_reviews[i].split() if word.isalnum() and word not in stop_words ])

high_dictionary = Dictionary(high_texts) # Create a GenSimdictionary
high_dictionary.filter_extremes(no_below=2) 
high_corpus = [high_dictionary.doc2bow(text) for text in high_texts] # Create a GenSim corpus
high_lda = LdaModel(
    corpus=high_corpus, 
    id2word=high_dictionary, 
    num_topics=3
)

pyLDAvis.enable_notebook()
high_vis = pyLDAvis.gensim_models.prepare(high_lda, high_corpus, high_dictionary,mds='mmds')

pyLDAvis.display(high_vis)

In [ ]:
low_texts = []
for i in range(len(restaurant_df_topic_modeling['Review'])):
    if(restaurant_df_topic_modeling['Rating'][i]<=2 ):
        low_texts.append([word for word in lemmatized_reviews[i].split() if word.isalnum() and word not in stop_words ])

low_dictionary = Dictionary(low_texts) # Create a GenSimdictionary
low_dictionary.filter_extremes(no_below=2) 
low_corpus = [low_dictionary.doc2bow(text) for text in low_texts] # Create a GenSim corpus
low_lda = LdaModel(
    corpus=low_corpus, 
    id2word=low_dictionary, 
    num_topics=3,
    random_state=100
)

pyLDAvis.enable_notebook()
low_vis = pyLDAvis.gensim_models.prepare(low_lda, low_corpus, low_dictionary,mds='mmds')

pyLDAvis.display(low_vis)

In [ ]:
#General topic modeling

texts = []
for i in range(len(restaurant_df_topic_modeling['Review'])):
    texts.append([word for word in lemmatized_reviews[i].split() if word.isalnum() and word not in stop_words ])

dictionary = Dictionary(high_texts) # Create a GenSimdictionary
dictionary.filter_extremes(no_below=2) 
corpus = [dictionary.doc2bow(text) for text in high_texts] # Create a GenSim corpus
lda = LdaModel(
    corpus=corpus, 
    id2word=dictionary, 
    num_topics=3
)

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda, corpus, dictionary,mds='mmds')

pyLDAvis.display(vis)